# Explorare: feed-ul GTFS, orarul de vârf și liniile alese (P4)

Notebook pentru etapele M1 și M2. Dacă feed-ul nu e încă în `data/raw/gtfs/`, se folosește feed-ul minuscul de test din `tests/fixtures/`.

In [ ]:
import os, sys
from pathlib import Path
# notebook-ul se poate deschide din notebooks/ sau din rădăcină; lucrăm mereu din rădăcină
if not Path("config/default.yaml").exists() and Path("../config/default.yaml").exists():
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sada.config import load_config
cfg = load_config("config/default.yaml")
print("Lucrez în", Path.cwd())

In [ ]:
from sada.p4.gtfs_io import load_feed, active_service_ids, route_mode, feed_date_range, time_to_seconds
gdir = Path(cfg["gtfs"]["dir"])
feed = load_feed(gdir if (gdir / "stops.txt").exists() else "tests/fixtures/mini_gtfs")
if not (gdir / "stops.txt").exists():
    print("Folosesc feed-ul DE TEST. Pentru cel real: python scripts/download_gtfs.py")
print("Valabil:", feed_date_range(feed))
routes = feed["routes"].assign(mod=lambda d: d.route_type.map(route_mode))
routes.groupby("mod").size()

## 1. Câte plecări pe oră, într-o zi lucrătoare

Vârful de dimineață trebuie să se vadă. Dacă ziua aleasă nu are servicii active, alegeți alta din perioada de valabilitate.

In [ ]:
date = cfg["gtfs"]["date"]
sids = active_service_ids(feed, date)
print("Servicii active:", sorted(sids)[:10], "...")
trips = feed["trips"][feed["trips"].service_id.isin(sids)]
st = feed["stop_times"][feed["stop_times"].trip_id.isin(set(trips.trip_id))].copy()
st["seq"] = st.stop_sequence.astype(int)
first = st.sort_values(["trip_id", "seq"]).groupby("trip_id").first()
dep_h = (time_to_seconds(first.departure_time) // 3600).astype(int)
dep_h.value_counts().sort_index().plot.bar()
plt.xlabel("Ora plecării din capăt"); plt.ylabel("Curse"); plt.title(f"Plecări pe oră, {date}"); plt.show()

## 2. Intervalele în vârf, pe linie

Liniile cu interval mare în vârf și care trec prin zone periferice sunt candidate bune pentru setul de 10–15 linii.

In [ ]:
from sada.p4.baseline import trips_in_window, headways
t = trips_in_window(feed, sids, *cfg["gtfs"]["window"])
hw = headways(t, window_s=7200).sort_values("headway_min")
display(hw.head(30))

## 3. Stațiile pe hartă, în metri (Stereo 70)

In [ ]:
from sada.p4.gtfs_io import stops_xy
xy = stops_xy(feed)
plt.figure(figsize=(6, 6)); plt.scatter(xy.x_m, xy.y_m, s=3)
plt.gca().set_aspect("equal"); plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.title(f"{len(xy)} stații"); plt.show()

## Ce notați pentru nota de proiectare

- ziua aleasă și de ce;
- cele 10–15 linii (cu tramvai, troleibuz, autobuz și cel puțin o linie spre o zonă cu acces slab);
- intervalele și orele-vehicul din S0 (`scripts/baseline_s0.py`);
- ce presupuneți despre timpul de întoarcere la capăt.